# 📈 FolioPP: Entry-Only Strategy Terminal (INDmoney)

This version identifies **Fresh Entry Triggers** (Long/Short) based on the **Institutional CUSUM/Kalman** core, with NO 'close' signals recorded.

In [69]:
# === 1. CONFIGURE PARAMETERS ===
SYMBOL = "RELIANCE"
DAYS = 365
INTERVAL = "1day"
EXCHANGE = "NSE"

print(f" Target: {SYMBOL} ({EXCHANGE})")

 Target: RELIANCE (NSE)


In [72]:
import os
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from dotenv import load_dotenv
from pykalman import KalmanFilter
import warnings 
warnings.filterwarnings("ignore")

load_dotenv("../.env")
TOKEN = os.getenv("INDSTOCKS_TOKEN") or os.getenv("INDMONEY_ACCESS_TOKEN")
BASE_URL = "https://api.indstocks.com"

def fetch_indmoney_data():
    headers = {"Authorization": TOKEN, "Content-Type": "application/json", "Accept": "*/*"}
    resp = requests.get(f"{BASE_URL}/market/instruments?source=equity", headers=headers)
    df_map = pd.read_csv(StringIO(resp.text))
    match = df_map[(df_map['TRADING_SYMBOL'] == SYMBOL) & (df_map['EXCH'] == EXCHANGE)]
    if match.empty: return None
    scrip = f"{EXCHANGE}_{match.iloc[0]['SECURITY_ID']}"
    
    end_ms = int(datetime.now().timestamp() * 1000)
    start_ms = end_ms - (DAYS * 24 * 60 * 60 * 1000)
    resp = requests.get(f"{BASE_URL}/market/historical/{INTERVAL}", headers=headers, params={"scrip-codes": scrip, "start_time": start_ms, "end_time": end_ms})
    data = resp.json().get("data", {}).get(scrip, {}).get("candles", [])
    if not data: return None
    
    df = pd.DataFrame(data)
    if isinstance(data[0], list): df.columns = ['timestamp', 'open', 'high', 'low', 'close', 'volume']
    else: df = df.rename(columns={'ts': 'timestamp', 'o': 'open', 'h': 'high', 'l': 'low', 'c': 'close', 'v': 'volume'})
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms' if df['timestamp'].iloc[0] > 1e12 else 's')
    return df

raw_data = fetch_indmoney_data()

In [73]:
# === 3.STRATEGY ENGINE ===

def process_data(data):
    class Indicators:
        data = None  
        @classmethod
        def initialize(cls, data: pd.DataFrame):
            cls.data = data.copy()
        @classmethod
        def calculate_indicators(cls):
            df = cls.data
            df['prev_close'] = df['close'].shift(1)
            df['prev_high'] = df['high'].shift(1)
            df['prev_low'] = df['low'].shift(1)
            df.dropna(subset=['prev_close'], inplace=True)
            
            # 1. Kalman Price
            prices = df['prev_close'].values.reshape(-1, 1)
            kf = KalmanFilter(transition_matrices=[1], observation_matrices=[1], initial_state_mean=prices[0], initial_state_covariance=1, observation_covariance=0.1, transition_covariance=0.01)
            state_means, _ = kf.filter(prices)
            df['kalman_price'] = pd.Series(state_means.flatten(), index=df.index)
            
            # 2. RSI (14)
            delta = df['prev_close'].diff()
            gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
            loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
            df['rsi_14'] = 100 - (100 / (1 + (gain/loss)))
            
            # 3. MACD Histogram
            exp1 = df['prev_close'].ewm(span=12, adjust=False).mean(); exp2 = df['prev_close'].ewm(span=26, adjust=False).mean()
            df['macd_hist'] = (exp1 - exp2) - (exp1 - exp2).ewm(span=9, adjust=False).mean()
            
            # 4. Regime Switch (CUSUM)
            rolling_sigma = df['prev_close'].rolling(window=5).std()
            price = df['prev_close'].values; mu = df['kalman_price'].values; k = 0.5 * rolling_sigma.fillna(0)
            S_hi = np.zeros(len(df)); S_lo = np.zeros(len(df))
            for i in range(1, len(df)):
                S_hi[i] = max(0, S_hi[i-1] + (price[i] - mu[i] - k.iloc[i]))
                S_lo[i] = max(0, S_lo[i-1] + (-price[i] + mu[i] - k.iloc[i]))
            df['regime'] = np.select([S_hi > rolling_sigma, S_lo > rolling_sigma], ['bullish', 'bearish'], default='No Trend')

    Indicators.initialize(data)
    Indicators.calculate_indicators()
    return Indicators.data

def strat(data):
    data['signals'] = 0; data['trade_type'] = " "
    # Non-Stateful Strategy: Only detects the START of a trend (Trigger Only)
    for i in range(1, len(data)):
        # Check for Bullish Entry Trigger
        if data['regime'].iloc[i] == 'bullish' and data['macd_hist'].iloc[i] > 0:
            # Only signal if it wasn't already long yesterday (FRESH TRIGGER)
            if not (data['regime'].iloc[i-1] == 'bullish' and data['macd_hist'].iloc[i-1] > 0):
                data.at[data.index[i], 'signals'] = 1; data.at[data.index[i], 'trade_type'] = 'long'
        
        # Check for Bearish Entry Trigger
        elif data['regime'].iloc[i] == 'bearish' and data['macd_hist'].iloc[i] < 0:
            # Only signal if it wasn't already short yesterday (FRESH TRIGGER)
            if not (data['regime'].iloc[i-1] == 'bearish' and data['macd_hist'].iloc[i-1] < 0):
                data.at[data.index[i], 'signals'] = -1; data.at[data.index[i], 'trade_type'] = 'short'
                
    return data

if raw_data is not None:
    print("Executing Trigger-Only Strategy...")
    results = strat(process_data(raw_data))
    filename = f"{SYMBOL}_{INTERVAL}_entry_triggers.csv"
    results.to_csv(filename, index=False)
    print(f" SUCCESS! Saved {len(results)} results to {filename}")
    display(results[['timestamp', 'prev_close', 'regime', 'macd_hist', 'trade_type']][results['trade_type'] != " "].tail(20))


Executing Trigger-Only Strategy...
 SUCCESS! Saved 247 results to RELIANCE_1day_entry_triggers.csv


,timestamp,prev_close,regime,macd_hist,trade_type
5,2025-04-02,1252.6,bearish,-2.649011,short
16,2025-04-22,1295.5,bullish,6.757602,long
62,2025-06-26,1467.3,bullish,0.061872,long
83,2025-07-25,1402.9,bearish,-14.688021,short
100,2025-08-20,1420.1,bullish,3.091856,long
108,2025-09-02,1353.9,bearish,-1.570558,short
116,2025-09-12,1383.3,bullish,2.292152,long
127,2025-09-29,1377.6,bearish,-0.629301,short
140,2025-10-17,1398.3,bullish,2.079339,long
193,2026-01-05,1592.3,bullish,0.662139,long
